In [2]:
import requests
import json
import re
import pandas as pd

In [3]:
def get_uniprot(accession):

    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"

    resp = requests.get(
        endpoint,
        headers={"Accept": "application/json"}
    )

    return resp

In [4]:
def uniprot_parse_response(resp):

    data = resp.json()
    accession = data["primaryAccession"]

    return {
        accession: {
            "organism": data["organism"]["scientificName"],
            "geneInfo": data.get("genes"),
            "sequenceInfo": data["sequence"],
            "type": "protein"
        }
    }

In [5]:
def get_ensembl(id):

    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"

    resp = requests.get(
        endpoint,
        headers={"Content-Type": "application/json"}
    )

    return resp

In [6]:
def ensembl_parse_response(resp):

    data = resp.json()
    ensembl_id = data["id"]

    return {
        ensembl_id: {
            "object_type": data.get("object_type"),
            "species": data.get("species"),
            "assembly_name": data.get("assembly_name"),
            "biotype": data.get("biotype"),
            "display_name": data.get("display_name"),
            "id": data.get("id"),
            "db_type": data.get("db_type"),
            "description": data.get("description"),
            "canonical_transcript": data.get("canonical_transcript"),
            "source": data.get("source")
        }
    }

In [7]:
def main(ids):

    results = {}

    for id in ids:

        try:

            if re.match(r"^[A-Z0-9]{6}$", id):

                resp = get_uniprot(id)

                if resp.status_code == 200:
                    parsed = uniprot_parse_response(resp)
                    results.update(parsed)
                else:
                    results[id] = "error:invalid uniprot id"

            elif re.match(r"^ENS[A-Z]*[0-9]+", id):

                resp = get_ensembl(id)

                if resp.status_code == 200:
                    parsed = ensembl_parse_response(resp)
                    results.update(parsed)
                else:
                    results[id] = "error:invalid ensembl id"

            else:
                results[id] = "error:unknown database"

        except Exception:
            results[id] = "error"

    return results

In [8]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'},
 'Q91XI3': {'organism': 'Ictidomys tridecemlineatus',
  'geneInfo': [{'geneName': {'value': 'INS'}}],
  'sequenceInfo': {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVE